In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

from options import OptionSurface, Deribit, OKX, Bybit
from portfolio_management import Portfolio
from api_client import TradingDeskAPI
from scanner import MarketScanner

In [2]:
# Initialize all classes and parameters

# load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
load_dotenv(r"C:/Users/brian/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
JWT_TOKEN = os.getenv("JWT")
DATA_DIR = os.getenv("DATA_DIR")
FRACTION = float(os.getenv("FRACTION"))
MAX_POSITION = float(os.getenv("MAX_POSITION"))
ENTRY_EV_THRESHOLD = float(os.getenv("ENTRY_EV_THRESHOLD")) # require 1% edge, default = 0
EXIT_EV_THRESHOLD = float(os.getenv("EXIT_EV_THRESHOLD"))
print(BASE_URL)

with open(f"{DATA_DIR}/crypto_tag_ids.json", "r") as f:
    crypto_tag_ids = set(json.load(f))

s = OptionSurface()
portfolio = Portfolio()
api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD, token=JWT_TOKEN)
scanner = MarketScanner(api=api)

currencies = ["BTC", "ETH"]

https://alphasignal-dev.moretoncp.com


In [3]:
# 1. Get all dfs
orders_df = pd.read_parquet(f"{DATA_DIR}/orders.parquet")
fills_df = pd.read_parquet(f"{DATA_DIR}/fills.parquet")
positions_df = pd.read_parquet(f"{DATA_DIR}/positions.parquet")
equity_df = pd.read_parquet(f"{DATA_DIR}/equity.parquet")

# fills_df = fills_df.iloc[0:0]

In [4]:
# 2. Get newly executed trades
fills_df = portfolio.sync_fills(api=api, fills_df=fills_df)
fills_df

,question,order_id,condition_id,token_id,outcome,side,price,shares,fee,timestamp,fill_id
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,BUY,0.981,4.87,0.00635,2026-08-14 08:25:11+00:00,4c3c68e2-7b89-463c-ae03-48b92808eeef
1,"Will Ethereum reach $4,500 by December 31, 2026?",0x387e06d4de5bb3110d135297db63ce667c4a3ac0de7d...,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,BUY,0.960,4.95,0.01330,2026-08-19 08:00:45+00:00,ff80c017-a13d-43ee-85c0-35a2c289c1b6
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xf7ff5b16da1bb114dc0b34d7e83324beb8b85502f029...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,BUY,0.969,4.91,0.01032,2026-08-19 08:01:06+00:00,cd0ebe33-7e52-44c6-a022-466914d90934
3,"Will Ethereum reach $5,500 by December 31, 2026?",0x007e608ee1b924c6137bc7fe08a909950bc98b3cf9e7...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,40.00,0.08410,2026-08-29 03:53:04+00:00,9c890c35-132a-44b6-ba48-b2fe16061fd3
4,"Will Ethereum reach $6,000 by December 31, 2026?",0x439db8be33af7e7d3497392a9f722930b7285e696ea4...,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,BUY,0.960,40.00,0.10752,2026-08-29 03:54:04+00:00,dbcbca55-34bd-446b-8a76-02c65229d62f
5,"Will Ethereum reach $6,000 by December 31, 2026?",0x6aba1d2deb08b681a8db29c758293485c023f541cb55...,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,BUY,0.960,46.00,0.12364,2026-08-29 04:02:15+00:00,0f6bafb7-aab6-49bf-a864-eb7be2c5ab5f
6,"Will Ethereum reach $5,500 by December 31, 2026?",0x1bb46bc40417ee40111bd4f3ef64b903fdecec03f2c2...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,35.00,0.07359,2026-08-29 04:03:17+00:00,567da504-7b00-4795-8543-c02e0bce9e0b
7,"Will Ethereum reach $5,500 by December 31, 2026?",0x1bb46bc40417ee40111bd4f3ef64b903fdecec03f2c2...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,11.00,0.00000,2026-08-29 09:06:59+00:00,69825e95-8228-4413-ac3b-60c9da86ec17


In [5]:
# 3. Reconstruct portfolio
# 4. Calculate realized P&L
positions_df = portfolio.reconstruct_positions_fifo(fills_df=fills_df)
positions_df

,question,condition_id,token_id,outcome,shares,cost_basis,avg_entry_price,realized_pnl,realized_shares,realized_fees
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,4.87,4.78382,0.982304,0.0,0.0,0.0
1,"Will Ethereum reach $4,500 by December 31, 2026?",0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,4.95,4.76530,0.962687,0.0,0.0,0.0
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,4.91,4.76811,0.971102,0.0,0.0,0.0
3,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,86.00,2.82369,0.032834,0.0,0.0,0.0
4,"Will Ethereum reach $6,000 by December 31, 2026?",0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,86.00,82.79116,0.962688,0.0,0.0,0.0


In [6]:
# 5. Sync positions with api
portfolio.reconcile_positions(api=api, positions_df=positions_df)
positions_df

POSITION SYNCED: 96993471854400156408670527613150944443359272190785251193551242374636006072800
POSITION SYNCED: 4251240413067872674686064123803499349976238079777545616008767482027755117695
POSITION SYNCED: 447913121087537662187327157218300928168985707076581166939650812231975265268
POSITION SYNCED: 61710247276022470252804012285430368172565001541144070158233974925141072188846
POSITION SYNCED: 99625432243305856023409516897537718601994293111805554742206464299633923525276


,question,condition_id,token_id,outcome,shares,cost_basis,avg_entry_price,realized_pnl,realized_shares,realized_fees
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,4.87,4.78382,0.982304,0.0,0.0,0.0
1,"Will Ethereum reach $4,500 by December 31, 2026?",0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,4.95,4.76530,0.962687,0.0,0.0,0.0
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,4.91,4.76811,0.971102,0.0,0.0,0.0
3,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,86.00,2.82369,0.032834,0.0,0.0,0.0
4,"Will Ethereum reach $6,000 by December 31, 2026?",0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,86.00,82.79116,0.962688,0.0,0.0,0.0


In [7]:
# 6. Get latest order state
orders_df = portfolio.sync_orders(api=api, orders_df=orders_df, fills_df=fills_df)
orders_df

,question,order_id,condition_id,token_id,outcome,side,price,requested_size,order_type,status,created_at,cancelled_at,filled_size,remaining_size
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,BUY,0.981,4.87,GTC,FILLED,2026-08-14 09:28:01.978131+00:00,NaT,4.87,0.0
1,"Will Ethereum reach $4,500 by December 31, 2026?",0x387e06d4de5bb3110d135297db63ce667c4a3ac0de7d...,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,BUY,0.960,4.95,GTC,FILLED,2026-08-19 08:00:40.748876+00:00,NaT,4.95,0.0
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xf7ff5b16da1bb114dc0b34d7e83324beb8b85502f029...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,BUY,0.969,4.91,GTC,FILLED,2026-08-19 08:01:02.498064+00:00,NaT,4.91,0.0
3,"Will Ethereum reach $5,500 by December 31, 2026?",0x007e608ee1b924c6137bc7fe08a909950bc98b3cf9e7...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,40.00,GTC,FILLED,2026-08-29 03:53:04+00:00,NaT,40.00,0.0
4,"Will Ethereum reach $6,000 by December 31, 2026?",0x439db8be33af7e7d3497392a9f722930b7285e696ea4...,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,BUY,0.960,40.00,GTC,FILLED,2026-08-29 03:54:04+00:00,NaT,40.00,0.0
5,"Will Ethereum reach $6,000 by December 31, 2026?",0x6aba1d2deb08b681a8db29c758293485c023f541cb55...,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,BUY,0.960,46.00,GTC,FILLED,2026-08-29 04:02:15+00:00,NaT,46.00,0.0
6,"Will Ethereum reach $5,500 by December 31, 2026?",0x1bb46bc40417ee40111bd4f3ef64b903fdecec03f2c2...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,46.00,GTC,FILLED,2026-08-29 04:03:17+00:00,NaT,46.00,0.0


In [8]:
# 7. Mark positions to market
positions_df = portfolio.mark_positions_to_market(api=api, positions_df=positions_df)
positions_df

,question,condition_id,token_id,outcome,shares,cost_basis,avg_entry_price,realized_pnl,realized_shares,realized_fees,current_price,market_value,unrealized_pnl,unrealized_return
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,4.87,4.78382,0.982304,0.0,0.0,0.0,0.985,4.79695,0.01313,0.002745
1,"Will Ethereum reach $4,500 by December 31, 2026?",0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,4.95,4.76530,0.962687,0.0,0.0,0.0,0.910,4.50450,-0.26080,-0.054729
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,4.91,4.76811,0.971102,0.0,0.0,0.0,0.960,4.71360,-0.05451,-0.011432
3,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,86.00,2.82369,0.032834,0.0,0.0,0.0,0.030,2.58000,-0.24369,-0.086302
4,"Will Ethereum reach $6,000 by December 31, 2026?",0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,86.00,82.79116,0.962688,0.0,0.0,0.0,0.961,82.64600,-0.14516,-0.001753


In [9]:
# 8. Calculate equity
equity_df = portfolio.calculate_equity(api=api, positions_df=positions_df, equity_df=equity_df)
equity_df

Equity:  99.34
Return:  -0.0049%
Sharpe:  -0.2092
Sortino: -0.1694


,timestamp,cash,market_value,equity,realized_pnl,unrealized_pnl,period_return,sharpe,sortino
0,2026-08-14 07:47:24.047885+00:00,100.00000,0.00000,100.00000,0.0,0.00000,NaN,NaN,NaN
1,2026-08-15 14:14:05.272655+00:00,95.21618,4.79695,100.01313,0.0,0.01948,0.000131,NaN,NaN
2,2026-08-17 02:58:57.343900+00:00,95.21618,4.77747,99.99365,0.0,0.00000,-0.000195,NaN,NaN
3,2026-08-19 07:56:33.538278+00:00,95.21618,4.78234,99.99852,0.0,0.00487,0.000049,NaN,NaN
4,2026-08-19 08:05:37.263592+00:00,85.68277,14.23772,99.92049,0.0,-0.04954,-0.000780,NaN,NaN
5,2026-08-24 13:32:14.011333+00:00,85.68747,13.80372,99.49119,0.0,-0.51351,-0.004296,NaN,NaN
6,2026-08-25 03:29:31.510439+00:00,85.68857,13.71458,99.40315,0.0,-0.60265,-0.000885,NaN,NaN
7,2026-08-27 02:38:40.684140+00:00,85.69107,13.69973,99.39080,0.0,-0.61750,-0.000124,NaN,NaN
8,2026-08-28 08:23:58.626786+00:00,85.69227,13.75861,99.45088,0.0,-0.55862,0.000604,NaN,NaN
9,2026-08-29 06:23:37.062020+00:00,0.41962,98.38844,98.80806,0.0,-1.20264,-0.006464,-0.553123,-0.414451


In [10]:
# 9. Get current markets
all_markets_df = api.get_all_markets(DATA_DIR=DATA_DIR, count_limit=10, liquidity_num_min=10000, volume_num_min=5000)
all_markets_df.head()

Fetched 100 markets | Total: 100
Fetched 100 markets | Total: 200
Fetched 100 markets | Total: 300
Fetched 100 markets | Total: 400
Fetched 100 markets | Total: 500
Fetched 100 markets | Total: 600
Fetched 100 markets | Total: 700
Fetched 100 markets | Total: 800
Fetched 100 markets | Total: 900
Fetched 100 markets | Total: 1,000


,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,version,negRiskMarketID,seriesColor,showGmpSeries,showGmpOutcome,oneHourPriceChange,umaResolutionStatus,eventStartTime,gameStartTime,groupItemRange
0,559651,Xi Jinping out before 2027?,0xa467b14d51f01b957109d9cbb1d6c124fab2a089d52e...,xi-jinping-out-before-2027,,2027-01-01T04:59:00Z,183786.76304,2025-07-03T20:37:00.228Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN
1,559652,Will Gavin Newsom win the 2028 Democratic pres...,0x0f49db97f71c68b1e42a6d16e3de93d85dbf7d4148e3...,will-gavin-newsom-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,338817.73073,2025-07-11T18:35:56.805Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
2,559653,Will Alexandria Ocasio-Cortez win the 2028 Dem...,0xe6bcc2f1dd025ce5e1833190f7c60a71171c94f805df...,will-alexandria-ocasio-cortez-win-the-2028-dem...,,2028-11-07T00:00:00Z,321580.80931,2025-07-11T18:35:59.075Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,-0.0010,NaN,NaN,NaN,NaN
3,559654,Will Pete Buttigieg win the 2028 Democratic pr...,0x4c325469d9b516ef4e6b8f73a81a12607dec075e3c2f...,will-pete-buttigieg-win-the-2028-democratic-pr...,,2028-11-07T00:00:00Z,350183.89421,2025-07-11T18:35:58.818Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
4,559655,Will Josh Shapiro win the 2028 Democratic pres...,0xd65891729ce093cc12236856837eba1a0872fc7998fd...,will-josh-shapiro-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,448861.90587,2025-07-11T18:36:01.098Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,-0.0005,NaN,NaN,NaN,NaN


In [11]:
all_markets_df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,version,negRiskMarketID,seriesColor,showGmpSeries,showGmpOutcome,oneHourPriceChange,umaResolutionStatus,eventStartTime,gameStartTime,groupItemRange
0,559651,Xi Jinping out before 2027?,0xa467b14d51f01b957109d9cbb1d6c124fab2a089d52e...,xi-jinping-out-before-2027,,2027-01-01T04:59:00Z,183786.76304,2025-07-03T20:37:00.228Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN
1,559652,Will Gavin Newsom win the 2028 Democratic pres...,0x0f49db97f71c68b1e42a6d16e3de93d85dbf7d4148e3...,will-gavin-newsom-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,338817.73073,2025-07-11T18:35:56.805Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
2,559653,Will Alexandria Ocasio-Cortez win the 2028 Dem...,0xe6bcc2f1dd025ce5e1833190f7c60a71171c94f805df...,will-alexandria-ocasio-cortez-win-the-2028-dem...,,2028-11-07T00:00:00Z,321580.80931,2025-07-11T18:35:59.075Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,-0.0010,NaN,NaN,NaN,NaN
3,559654,Will Pete Buttigieg win the 2028 Democratic pr...,0x4c325469d9b516ef4e6b8f73a81a12607dec075e3c2f...,will-pete-buttigieg-win-the-2028-democratic-pr...,,2028-11-07T00:00:00Z,350183.89421,2025-07-11T18:35:58.818Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
4,559655,Will Josh Shapiro win the 2028 Democratic pres...,0xd65891729ce093cc12236856837eba1a0872fc7998fd...,will-josh-shapiro-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,448861.90587,2025-07-11T18:36:01.098Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,-0.0005,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1235548,Will the Toronto Blue Jays win the 2026 World ...,0x6df5c681e14dadf57c0ba0a7c01d6e2cb04f0f97528e...,will-the-toronto-blue-jays-win-the-2026-world-...,,2026-10-31T23:55:00Z,75552.99511,2026-01-21T20:45:03.862Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x85de20785f24956ff4ce1471e80f7d379d97f3fb8a20...,,False,False,NaN,NaN,NaN,NaN,NaN
996,1235549,Will the Tampa Bay Rays win the 2026 World Ser...,0x4d9567b9fa71a94b6e43ddb62ea6104cbb0192681393...,will-the-tampa-bay-rays-win-the-2026-world-series,,2026-10-31T23:55:00Z,118272.41961,2026-01-21T20:45:04.651Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x85de20785f24956ff4ce1471e80f7d379d97f3fb8a20...,,False,False,NaN,NaN,NaN,NaN,NaN
997,1235550,Will the Baltimore Orioles win the 2026 World ...,0xae8cf6ba7bd936d12c4c77b77b800dc378a76f02d027...,will-the-baltimore-orioles-win-the-2026-world-...,,2026-10-31T23:55:00Z,108818.38277,2026-01-21T20:45:05.165061Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x85de20785f24956ff4ce1471e80f7d379d97f3fb8a20...,,False,False,NaN,NaN,NaN,NaN,NaN
998,1235551,Will the Boston Red Sox win the 2026 World Ser...,0x3e38887e936b8fedd5373852777e7a3e90805affd813...,will-the-boston-red-sox-win-the-2026-world-series,,2026-10-31T23:55:00Z,110609.5125,2026-01-21T20:45:04.907Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x85de20785f24956ff4ce1471e80f7d379d97f3fb8a20...,,False,False,NaN,NaN,NaN,NaN,NaN


In [12]:
# 10. Initialize variance surface
deribit = Deribit(currencies=currencies)
okx = OKX(currencies=currencies)
bybit = Bybit(currencies=currencies)

s.initialize(currencies=currencies, exchanges=[deribit, okx, bybit])

currency: BTC, spot: 78419.0, volume24h: 1.8449
currency: ETH, spot: 2467.6, volume24h: 49.2989
currency: BTC, spot: 78736.2, volume24h: 5240.78700331
currency: ETH, spot: 2471.15, volume24h: 92585.910768
currency: BTC, spot: 78744.9, volume24h: 5406.978021
currency: ETH, spot: 2471.09, volume24h: 57090.84583


In [14]:
# 11. Scan markets
markets_df, opportunities_df, arb_candidates_df = scanner.scan_market(markets_df=all_markets_df, s=s, crypto_tag_ids=crypto_tag_ids)
opportunities_df.head()

question: Will Bitcoin hit $150k by December 31, 2026?
event type: touch
direction: up
currency: BTC
required strike: 150000.0
iv: 0.5004801387106533
buy_yes_ev: -0.01822743339481235
sell_yes_ev: 0.012694583394812352
buy_no_ev: 0.012694583394812355
sell_no_ev: -0.018227433394812385
buy_yes_kelly: -0.009456996987619537
sell_yes_kelly: 0.2063071002656541
buy_no_kelly: 0.20630710026565413
sell_no_kelly: -0.009456996987619554

question: Will Bitcoin reach $200,000 by December 31, 2026?
event type: touch
direction: up
currency: BTC
required strike: 200000.0
iv: 0.583951608053533
buy_yes_ev: -0.011550907883265779
sell_yes_ev: 0.005891627883265777
buy_no_ev: 0.005891627883265732
sell_no_ev: -0.011550907883265776
buy_yes_kelly: -0.005863204298866936
sell_yes_kelly: 0.3165159494609314
buy_no_kelly: 0.31651594946093053
sell_no_kelly: -0.005863204298866934

question: Will Bitcoin reach $190,000 by December 31, 2026?
event type: touch
direction: up
currency: BTC
required strike: 190000.0
iv: 0.579

,question,endDate,yes_ask,yes_bid,no_ask,no_bid,model_prob,id,conditionId,slug,...,buy_yes_ev,sell_yes_ev,buy_no_ev,sell_no_ev,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
583,"Will Ethereum dip to $1,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.142,0.141,0.859,0.858,0.198734,701552,0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...,will-ethereum-dip-to-1500-by-december-31-2026-...,...,0.048206,-0.066213,-0.066213,0.048206,0.028374,-0.249818,-0.249818,0.028374,0.048206,buy_yes_ev
581,"Will Ethereum reach $4,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.15,0.14,0.86,0.85,0.109792,701548,0x9775cd557a3cdba4f0478070afa399c0805761dd9aa0...,will-ethereum-reach-4000-by-december-31-2026,...,-0.049133,0.021780,0.021780,-0.049133,-0.029209,0.082770,0.082770,-0.029209,0.021780,sell_yes_ev
569,"Will Bitcoin dip to $45,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.08,0.07,0.93,0.92,0.104262,701502,0x024b68f77bfc019341ee3db8f57c103334e4b9430bba...,will-bitcoin-dip-to-45000-by-december-31-2026-...,...,0.019110,-0.038819,-0.038819,0.019110,0.010445,-0.296589,-0.296589,0.010445,0.019110,buy_yes_ev
576,"Will Ethereum reach $6,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.038,0.029,0.971,0.962,0.008126,701543,0x0f0499d1049385b1d53ffee6c42a1de7424e551c7652...,will-ethereum-reach-6500-by-december-31-2026,...,-0.032433,0.018903,0.018903,-0.032433,-0.016902,0.349676,0.349676,-0.016902,0.018903,sell_yes_ev
579,"Will Ethereum reach $5,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.072,0.054,0.946,0.928,0.033671,701546,0x1c4fd67ab2a67f508672a69153559911244048b79a40...,will-ethereum-reach-5000-by-december-31-2026,...,-0.043006,0.016753,0.016753,-0.043006,-0.023289,0.166118,0.166118,-0.023289,0.016753,buy_no_ev


In [ ]:
# 3	    Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.142	0.141	0.859	0.858	0.198734	701552	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...	...	0.048206	-0.066213	-0.066213	0.048206	0.028374	-0.249818	-0.249818	0.028374	0.048206	buy_yes_ev

# 760	Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.15	0.146	0.854	0.85	0.209409	701552	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...	...	0.050484	-0.072136	-0.072136	0.050484	0.030011	-0.262750	-0.262750	0.030011	0.050484	buy_yes_ev

# 815	Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.172	0.155	0.845	0.828	0.221717	701552	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...	...	0.039748	-0.075885	-0.075885	0.039748	0.024295	-0.260180	-0.260180	0.024295	0.039748	buy_yes_ev

#       Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.122	0.121	0.879	0.878	0.219853	701552	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...	...	0.090355	-0.106299	-0.106299	0.090355	0.051898	-0.468050	-0.468050	0.051898	0.090355	buy_yes_ev

# 06	Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.146	0.89	0.264426	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...		59292.16479	...	0.109698	-0.161279	-0.161279	0.109698	0.064889	-0.781790	-0.781790	0.064889	0.109698	buy_yes_ev

# 585	Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.402	0.599	0.525600	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...		80206.24887	...	0.106772	-0.141414	-0.141414	0.106772	0

In [16]:
opportunities_df

,question,endDate,yes_ask,yes_bid,no_ask,no_bid,model_prob,id,conditionId,slug,...,buy_yes_ev,sell_yes_ev,buy_no_ev,sell_no_ev,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
583,"Will Ethereum dip to $1,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.142,0.141,0.859,0.858,0.198734,701552,0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...,will-ethereum-dip-to-1500-by-december-31-2026-...,...,0.048206,-0.066213,-0.066213,0.048206,0.028374,-0.249818,-0.249818,0.028374,0.048206,buy_yes_ev
581,"Will Ethereum reach $4,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.15,0.14,0.86,0.85,0.109792,701548,0x9775cd557a3cdba4f0478070afa399c0805761dd9aa0...,will-ethereum-reach-4000-by-december-31-2026,...,-0.049133,0.021780,0.021780,-0.049133,-0.029209,0.082770,0.082770,-0.029209,0.021780,sell_yes_ev
569,"Will Bitcoin dip to $45,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.08,0.07,0.93,0.92,0.104262,701502,0x024b68f77bfc019341ee3db8f57c103334e4b9430bba...,will-bitcoin-dip-to-45000-by-december-31-2026-...,...,0.019110,-0.038819,-0.038819,0.019110,0.010445,-0.296589,-0.296589,0.010445,0.019110,buy_yes_ev
576,"Will Ethereum reach $6,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.038,0.029,0.971,0.962,0.008126,701543,0x0f0499d1049385b1d53ffee6c42a1de7424e551c7652...,will-ethereum-reach-6500-by-december-31-2026,...,-0.032433,0.018903,0.018903,-0.032433,-0.016902,0.349676,0.349676,-0.016902,0.018903,sell_yes_ev
579,"Will Ethereum reach $5,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.072,0.054,0.946,0.928,0.033671,701546,0x1c4fd67ab2a67f508672a69153559911244048b79a40...,will-ethereum-reach-5000-by-december-31-2026,...,-0.043006,0.016753,0.016753,-0.043006,-0.023289,0.166118,0.166118,-0.023289,0.016753,buy_no_ev
580,"Will Ethereum reach $4,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.09,0.08,0.92,0.91,0.058205,701547,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,will-ethereum-reach-4500-by-december-31-2026,...,-0.037528,0.016643,0.016643,-0.037528,-0.020751,0.111180,0.111180,-0.020751,0.016643,sell_yes_ev
577,"Will Ethereum reach $6,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.039,0.031,0.969,0.961,0.013068,701544,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,will-ethereum-reach-6000-by-december-31-2026,...,-0.028556,0.015830,0.015830,-0.028556,-0.014898,0.273896,0.273896,-0.014898,0.015830,sell_yes_ev
582,"Will Ethereum reach $3,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.27,0.25,0.75,0.73,0.221983,701549,0x42945ea5657d6f6e77969a06661b29d6f6295083d1ef...,will-ethereum-reach-3500-by-december-31-2026,...,-0.061814,0.014892,0.014892,-0.061814,-0.043154,0.031434,0.031434,-0.043154,0.014892,sell_yes_ev
573,"Will Ethereum reach $8,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.02,0.019,0.981,0.98,0.002946,701540,0xac0b4316113b50e0e667918aee72fa0eb006342245a5...,will-ethereum-reach-8000-by-december-31-2026,...,-0.018426,0.014749,0.014749,-0.018426,-0.009414,0.416753,0.416753,-0.009414,0.014749,buy_no_ev
567,"Will Bitcoin reach $100,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.24,0.23,0.77,0.76,0.267477,701496,0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49db...,will-bitcoin-reach-100000-by-december-31-2026-...,...,0.014709,-0.049874,-0.049874,0.014709,0.009842,-0.114598,-0.114598,0.009842,0.014709,buy_yes_ev


In [17]:
cross_market_arb_df, vertical_arb_df = scanner.scan_arbitrage(arb_candidates_df)
vertical_arb_df

                                              question               endDate  \
165       Will Bitcoin hit $150k by December 31, 2026?  2027-01-01T05:00:00Z   
562  Will Bitcoin reach $150,000 by December 31, 2026?  2027-01-01T05:00:00Z   

    yes_ask yes_bid no_ask no_bid  model_prob      id  \
165   0.034   0.033  0.967  0.966    0.018072  573656   
562   0.032   0.029  0.971  0.968    0.018072  701491   

                                           conditionId  \
165  0x02deb9538f5c123373adaa4ee6217b01745f1662bc90...   
562  0xa7b594ae07d5c1590fa86028fcc2f870599043723741...   

                                                  slug  ... buy_yes_ev  \
165          will-bitcoin-hit-150k-by-december-31-2026  ...  -0.018227   
562  will-bitcoin-reach-150000-by-december-31-2026-...  ...  -0.016097   

    sell_yes_ev buy_no_ev sell_no_ev buy_yes_kelly sell_yes_kelly  \
165    0.012695  0.012695  -0.018227     -0.009457       0.206307   
562    0.008957  0.008957  -0.016097     -0.008333 

,currency,event_type,direction,lower_strike,lower_question,lower_id,lower_side,lower_price,lower_cost,higher_strike,higher_question,higher_id,higher_side,higher_price,higher_cost,total_cost,guaranteed_profit


In [18]:
# 12. manage cancel orders
orders_df = portfolio.manage_open_orders(api=api, orders_df=orders_df, markets_df=markets_df, 
                ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)
orders_df

,question,order_id,condition_id,token_id,outcome,side,price,requested_size,order_type,status,created_at,cancelled_at,filled_size,remaining_size
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,BUY,0.981,4.87,GTC,FILLED,2026-08-14 09:28:01.978131+00:00,NaT,4.87,0.0
1,"Will Ethereum reach $4,500 by December 31, 2026?",0x387e06d4de5bb3110d135297db63ce667c4a3ac0de7d...,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,BUY,0.960,4.95,GTC,FILLED,2026-08-19 08:00:40.748876+00:00,NaT,4.95,0.0
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xf7ff5b16da1bb114dc0b34d7e83324beb8b85502f029...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,BUY,0.969,4.91,GTC,FILLED,2026-08-19 08:01:02.498064+00:00,NaT,4.91,0.0
3,"Will Ethereum reach $5,500 by December 31, 2026?",0x007e608ee1b924c6137bc7fe08a909950bc98b3cf9e7...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,40.00,GTC,FILLED,2026-08-29 03:53:04+00:00,NaT,40.00,0.0
4,"Will Ethereum reach $6,000 by December 31, 2026?",0x439db8be33af7e7d3497392a9f722930b7285e696ea4...,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,BUY,0.960,40.00,GTC,FILLED,2026-08-29 03:54:04+00:00,NaT,40.00,0.0
5,"Will Ethereum reach $6,000 by December 31, 2026?",0x6aba1d2deb08b681a8db29c758293485c023f541cb55...,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,BUY,0.960,46.00,GTC,FILLED,2026-08-29 04:02:15+00:00,NaT,46.00,0.0
6,"Will Ethereum reach $5,500 by December 31, 2026?",0x1bb46bc40417ee40111bd4f3ef64b903fdecec03f2c2...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,46.00,GTC,FILLED,2026-08-29 04:03:17+00:00,NaT,46.00,0.0


In [19]:
# 13. Risk management
orders_df = portfolio.run_risk_management(api=api, positions_df=positions_df, 
            markets_df=markets_df, orders_df=orders_df, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

pos: question             Will Bitcoin reach $200,000 by December 31, 2026?
condition_id         0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...
token_id             9699347185440015640867052761315094444335927219...
outcome                                                             No
shares                                                            4.87
cost_basis                                                     4.78382
avg_entry_price                                               0.982304
realized_pnl                                                       0.0
realized_shares                                                    0.0
realized_fees                                                      0.0
current_price                                                    0.986
market_value                                                   4.80182
unrealized_pnl                                                   0.018
unrealized_return                                             0.003763
N

In [20]:
# 14. New opportunities
orders_df = portfolio.run_new_opportunities(api=api, opportunities_df=opportunities_df, orders_df=orders_df, 
            ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, MAX_POSITION=MAX_POSITION, FRACTION=FRACTION)

current balance size: 0.0
Order too small after position cap, skipping, dollars: 0.0007357363375212562
current balance size: 0.0
Order too small after position cap, skipping, dollars: 0.002146230782337518
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current trade is less than required ev (0.02), skipping
Current tra

In [ ]:
#     # 15. Safe dfs
portfolio.save_snapshots(DATA_DIR, markets_df, "markets")
portfolio.save_snapshots(DATA_DIR, opportunities_df, "opportunities")
portfolio.save(DATA_DIR, orders_df, "orders")
portfolio.save(DATA_DIR, fills_df, "fills")
portfolio.save(DATA_DIR, positions_df, "positions")
portfolio.save(DATA_DIR, equity_df, "equity")

save_snapshots df saved at: data/20260901/markets_20260901_125446.parquet
save_snapshots df saved at: data/20260901/opportunities_20260901_125446.parquet
save df saved at:  data/orders.parquet
save df saved at:  data/fills.parquet
save df saved at:  data/positions.parquet
save df saved at:  data/realized_pnl.parquet
save df saved at:  data/equity.parquet


In [ ]:
# Polymarket    Model (exchanges: deribit, bybit, okx)    Edge

# 80k Dec-26        22%          20%       +2%
# 90k Dec-26        14%          12%       +2%
# 100k Dec-26        9.5%         5.9%     +3.6%
# 110k Dec-26        7%           4%       +3%
# 120k Dec-26        5%           2%       +3%

In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

# 1. Is the options-derived probability actually predictive?

# 2. Are you accounting for crypto risk premia / risk-neutral vs physical probabilities?

# 3. Are your touch probabilities correctly calibrated?

# 4. Are fees and prediction-market spreads killing the apparent EV?

# 5. Does your exit rule actually improve realized P&L?

# 6. Are multiple contracts giving you the same underlying exposure?

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Yes — potentially a lot, but only if you structure it correctly. If you're being graded on Sortino, the important thing is not simply “arbitrage = good.” It's whether the strategy improves downside-adjusted returns relative to the capital and risk it consumes.

# For your specific setup, I'd think about it this way:

# 1. Why the vertical arbitrage can help Sortino

# Your 75 paired contracts:

# YES $5,500
# NO $6,000

# have a very constrained payoff structure. Once both legs are filled at sufficiently favorable prices, the pair has approximately:

# $$ \$1-(P_{5500,YES}+P_{6000,NO}) $$

# of gross locked-in value per pair.

# That is attractive from a Sortino perspective because you're converting some capital into a low-variance / low-downside-return stream.

# If the pair costs $0.991 and ultimately pays $1:

# $$ \text{return} \approx \frac{0.009}{0.991}=0.91\% $$

# before fees.

# After your stated fees, it's roughly:

# $$ \frac{0.00421}{0.991}\approx\boxed{0.42\%} $$

# per completed pair.

# The key word is completed. If you only buy one side and fail to acquire the other, you haven't created the low-downside arbitrage payoff—you've created directional exposure.

# 2. But there's a subtle Sortino issue

# If your grading period is short, an arbitrage trade that makes $0.30 over several weeks may actually hurt your Sortino if it ties up capital that could have generated larger returns elsewhere.

# Sortino essentially asks:

# How much return am I generating per unit of downside volatility?

# So you care about:

# $$ \text{Sortino} = \frac{R_p-R_{\text{target}}} {\text{downside deviation}} $$

# A low-risk arbitrage can improve the denominator dramatically, but only if its return is meaningful relative to the capital allocated and the evaluation horizon.

# 3. Your residual NO position is different

# This is where your interview explanation becomes interesting.

# Your 11 excess NO $6,000 contracts aren't part of the arbitrage. They're directional.

# So conceptually your book is:

# Core

# 75 matched vertical arbitrage pairs → low downside / low variance

# Satellite

# 11 excess NO $6,000 → directional model exposure

# That's actually a reasonable portfolio construction approach.

# You're not saying:

# “I accidentally have an imbalance.”

# You're saying:

# “I deliberately allowed a residual directional position because my model's expected value for that exposure justified retaining it, while using the rest of my capital for a lower-risk relative-value strategy.”

# That is a much stronger trading argument.

# 4. However, don't claim the arbitrage automatically improves Sortino

# An interviewer could challenge you:

# “Why not just allocate all your capital to the arbitrage?”

# Your answer needs to be:

# “Because I was optimizing portfolio-level downside-adjusted return, not maximizing the number of arbitrage contracts. Once I accounted for fees, execution risk, capital utilization, and the expected return of the existing directional position, the marginal arbitrage wasn't necessarily the highest-Sortino use of capital.”

# That's a sophisticated answer.

# The biggest thing I'd watch

# Your stated edge is only about 0.42¢ per pair after fees.

# That's extremely thin.

# At that level, execution risk may dominate the theoretical arbitrage edge. A 1-minute delay, partial fill, or a few ticks of adverse movement can erase the expected profit.

# So for an interview, I'd explicitly say:

# “I only consider the vertical a true arbitrage after both legs are executable at prices that leave positive expected edge after fees and execution costs. Otherwise, I treat the unhedged leg as directional risk.”

# That's probably the strongest way to present it.

# Bottom line: Yes, the strategy can improve your Sortino because it can add positive, relatively low-downside P&L. But your portfolio construction and execution discipline are what make it Sortino-positive—not merely labeling the trade “arbitrage.”

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4